# ADMM TEST

In [ ]:
import time
import struct
import numpy as np
import scipy.sparse as sp
from pynq import Overlay, allocate, MMIO

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================
# IMPORTANT: Update this path to your actual bitstream location
bitstream_path = '/home/xilinx/admm/admm.bit' 
print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print('Overlay loaded!')

CONTROL_ADDR   = 0xA0000000
CONTROL_R_ADDR = 0xA0010000
ADDR_RANGE     = 0x10000

print(f"Manually mapping Control   : {hex(CONTROL_ADDR)}")
print(f"Manually mapping Control_R : {hex(CONTROL_R_ADDR)}")

control_ip   = MMIO(CONTROL_ADDR, ADDR_RANGE)
control_r_ip = MMIO(CONTROL_R_ADDR, ADDR_RANGE)

In [ ]:
import os
import time
import struct
import numpy as np
import scipy.sparse as sp

from pynq import Overlay, allocate, MMIO

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================

bitstream_path = '/home/xilinx/admm/admm.bit'

print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print("Overlay loaded!")

CONTROL_ADDR = 0xA0000000
CONTROL_R_ADDR = 0xA0010000
ADDR_RANGE = 0x10000

control_ip = MMIO(CONTROL_ADDR, ADDR_RANGE)
control_r_ip = MMIO(CONTROL_R_ADDR, ADDR_RANGE)

# =====================================================================
# 2. Load Canonicalized CVXPY QP Data
# =====================================================================

data_dir = "./data"

print(f"\nLoading QP data from: {os.path.abspath(data_dir)}")

# Load sparse matrices
P_sparse = sp.load_npz(os.path.join(data_dir, "P.npz")).tocsc()
A_sparse = sp.load_npz(os.path.join(data_dir, "A.npz")).tocsc()

# Load vectors
q = np.load(os.path.join(data_dir, "q.npy")).astype(np.float32)
l_in = np.load(os.path.join(data_dir, "l.npy")).astype(np.float32)
u_in = np.load(os.path.join(data_dir, "u.npy")).astype(np.float32)

# Problem dimensions
num_rows, num_cols = A_sparse.shape

print(f"Problem size: {num_rows} x {num_cols}")

# =====================================================================
# 3. Extract CSC Data
# =====================================================================

PACK_SIZE = 16

# -------------------------
# A matrix
# -------------------------

A_col_ptr = A_sparse.indptr.astype(np.int32)
A_row_idx = A_sparse.indices.astype(np.int32)
A_values = A_sparse.data.astype(np.float32)

A_nnz = int(A_sparse.nnz)

# -------------------------
# A transpose
# -------------------------

AT_sparse = A_sparse.transpose().tocsc()

AT_col_ptr = AT_sparse.indptr.astype(np.int32)
AT_row_idx = AT_sparse.indices.astype(np.int32)
AT_values = AT_sparse.data.astype(np.float32)

# -------------------------
# P diagonal
# -------------------------

P_diag = P_sparse.diagonal().astype(np.float32)

P_values = P_diag.copy()
P_row_idx = np.arange(num_cols, dtype=np.int32)
P_col_ptr = np.arange(num_cols + 1, dtype=np.int32)

P_nnz = len(P_diag)

print(f"A nnz: {A_nnz}")
print(f"P nnz: {P_nnz}")

# =====================================================================
# 4. Solver Parameters
# =====================================================================

alpha = 1.6
sigma = 1e-6

admm_max_iterations = 4000
pcg_max_iterations = 200

RHO_INIT = 1e-1

rho_in = np.full(num_rows, RHO_INIT, dtype=np.float32)

RUN_BOTH_ADAPTIVE_RHO = True

# =====================================================================
# 5. Utility Functions
# =====================================================================

def ceil_div(a, b):
    return (a + b - 1) // b

def pack_csc_to_words(row_idx, values, num_words):
    nnz = len(row_idx)

    row_words = allocate(
        shape=(num_words, PACK_SIZE),
        dtype=np.int32,
        cacheable=False
    )

    val_words = allocate(
        shape=(num_words, PACK_SIZE),
        dtype=np.float32,
        cacheable=False
    )

    row_words[:] = 0
    val_words[:] = 0.0

    row_words.reshape(-1)[:nnz] = row_idx
    val_words.reshape(-1)[:nnz] = values

    return row_words, val_words

def write_64bit_address(ip, base_offset, address):
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)

def float_to_uint(f_val):
    return struct.unpack('<I', struct.pack('<f', float(f_val)))[0]

def uint_to_float(u_val):
    return struct.unpack('<f', struct.pack('<I', int(u_val)))[0]

def objective_val(x_vec):
    return float(
        0.5 * np.sum(P_diag * x_vec * x_vec)
        + np.dot(q, x_vec)
    )

def max_box_violation(z, l, u):
    return float(
        np.max(np.maximum(0.0, np.maximum(l - z, z - u)))
    )

# =====================================================================
# 6. Allocate Hardware Buffers
# =====================================================================

print("\nAllocating hardware buffers...")

A_words = int(ceil_div(A_nnz, PACK_SIZE))
AT_words = int(ceil_div(A_nnz, PACK_SIZE))
P_words = int(ceil_div(P_nnz, PACK_SIZE))

A_row_words, A_val_words = pack_csc_to_words(
    A_row_idx,
    A_values,
    A_words
)

AT_row_words, AT_val_words = pack_csc_to_words(
    AT_row_idx,
    AT_values,
    AT_words
)

P_row_words, P_val_words = pack_csc_to_words(
    P_row_idx,
    P_values,
    P_words
)

PAD = 16

# -----------------------------------------------------------------
# Pointer arrays
# -----------------------------------------------------------------

A_col_ptr_hw = allocate(
    shape=(num_cols + 1 + PAD,),
    dtype=np.int32,
    cacheable=False
)
A_col_ptr_hw[: num_cols + 1] = A_col_ptr

AT_col_ptr_hw = allocate(
    shape=(num_rows + 1 + PAD,),
    dtype=np.int32,
    cacheable=False
)
AT_col_ptr_hw[: num_rows + 1] = AT_col_ptr

P_col_ptr_hw = allocate(
    shape=(num_cols + 1 + PAD,),
    dtype=np.int32,
    cacheable=False
)
P_col_ptr_hw[: num_cols + 1] = P_col_ptr

# -----------------------------------------------------------------
# Dense vectors
# -----------------------------------------------------------------

P_diag_hw = allocate(
    shape=(num_cols + PAD,),
    dtype=np.float32,
    cacheable=False
)
P_diag_hw[:num_cols] = P_diag

l_in_hw = allocate(
    shape=(num_rows + PAD,),
    dtype=np.float32,
    cacheable=False
)
l_in_hw[:num_rows] = l_in

u_in_hw = allocate(
    shape=(num_rows + PAD,),
    dtype=np.float32,
    cacheable=False
)
u_in_hw[:num_rows] = u_in

q_in_hw = allocate(
    shape=(num_cols + PAD,),
    dtype=np.float32,
    cacheable=False
)
q_in_hw[:num_cols] = q

rho_in_hw = allocate(
    shape=(num_rows + PAD,),
    dtype=np.float32,
    cacheable=False
)
rho_in_hw[:num_rows] = rho_in

# -----------------------------------------------------------------
# Outputs
# -----------------------------------------------------------------

x_out_hw = allocate(
    shape=(num_cols + PAD,),
    dtype=np.float32,
    cacheable=False
)

y_out_hw = allocate(
    shape=(num_rows + PAD,),
    dtype=np.float32,
    cacheable=False
)

x_out_hw[:] = 0.0
y_out_hw[:] = 0.0

# =====================================================================
# 7. Hardware Execution
# =====================================================================

def run_hw(adaptive_rho):

    print(f"\n=== HW Run (adaptive_rho={adaptive_rho}) ===")

    # -------------------------------------------------------------
    # Control registers
    # -------------------------------------------------------------

    control_ip.write(0x10, num_rows)
    control_ip.write(0x18, num_cols)
    control_ip.write(0x20, A_nnz)
    control_ip.write(0x28, P_nnz)

    control_ip.write(0x30, float_to_uint(sigma))
    control_ip.write(0x38, float_to_uint(alpha))

    control_ip.write(0x40, admm_max_iterations)
    control_ip.write(0x48, pcg_max_iterations)

    control_ip.write(0x50, int(adaptive_rho))

    # -------------------------------------------------------------
    # Address registers
    # -------------------------------------------------------------

    write_64bit_address(control_r_ip, 0x10, A_row_words.device_address)
    write_64bit_address(control_r_ip, 0x1c, A_col_ptr_hw.device_address)
    write_64bit_address(control_r_ip, 0x28, A_val_words.device_address)

    write_64bit_address(control_r_ip, 0x34, AT_row_words.device_address)
    write_64bit_address(control_r_ip, 0x40, AT_col_ptr_hw.device_address)
    write_64bit_address(control_r_ip, 0x4c, AT_val_words.device_address)

    write_64bit_address(control_r_ip, 0x58, P_row_words.device_address)
    write_64bit_address(control_r_ip, 0x64, P_col_ptr_hw.device_address)
    write_64bit_address(control_r_ip, 0x70, P_val_words.device_address)

    write_64bit_address(control_r_ip, 0x7c, P_diag_hw.device_address)

    write_64bit_address(control_r_ip, 0x88, l_in_hw.device_address)
    write_64bit_address(control_r_ip, 0x94, u_in_hw.device_address)

    write_64bit_address(control_r_ip, 0xa0, q_in_hw.device_address)

    write_64bit_address(control_r_ip, 0xac, rho_in_hw.device_address)

    write_64bit_address(control_r_ip, 0xb8, x_out_hw.device_address)
    write_64bit_address(control_r_ip, 0xc4, y_out_hw.device_address)

    # -------------------------------------------------------------
    # Start accelerator
    # -------------------------------------------------------------

    x_out_hw[:] = 0.0
    y_out_hw[:] = 0.0

    hw_start = time.time()

    control_ip.write(0x00, 0x01)

    while (control_ip.read(0x00) & 0x02) == 0:
        pass

    hw_end = time.time()

    print(f"HW execution time: {(hw_end - hw_start) * 1000:.4f} ms")

    # -------------------------------------------------------------
    # Read results
    # -------------------------------------------------------------

    admm_iters = int(control_ip.read(0x58))
    pcg_iters = int(control_ip.read(0x68))

    r_prim_out = float(
        uint_to_float(control_ip.read(0x78))
    )

    r_dual_out = float(
        uint_to_float(control_ip.read(0x88))
    )

    status_out = int(control_ip.read(0x98))

    x_hw = np.array(
        x_out_hw[:num_cols],
        dtype=np.float32
    )

    Ax_hw = (A_sparse @ x_hw).astype(np.float32)

    viol_hw = max_box_violation(
        Ax_hw,
        l_in,
        u_in
    )

    obj_hw = objective_val(x_hw)

    # -------------------------------------------------------------
    # Print summary
    # -------------------------------------------------------------

    print("\n--- Results ---")

    print(
        f"Kernel Status: "
        f"{'Converged' if status_out == 1 else 'Max Iterations'}"
    )

    print(f"ADMM Iterations: {admm_iters}")
    print(f"PCG Iterations : {pcg_iters}")

    print(f"Primal Residual: {r_prim_out:.5e}")
    print(f"Dual Residual  : {r_dual_out:.5e}")

    print(f"Constraint Violation: {viol_hw:.3e}")

    print(f"Objective Value: {obj_hw:.6e}")

    return {
        "adaptive_rho": adaptive_rho,
        "status": status_out,
        "admm_iters": admm_iters,
        "pcg_iters": pcg_iters,
        "r_prim": r_prim_out,
        "r_dual": r_dual_out,
        "viol": viol_hw,
        "obj": obj_hw,
        "hw_ms": (hw_end - hw_start) * 1000.0,
    }

# =====================================================================
# 8. Run Tests
# =====================================================================

adaptive_rho_list = [0, 1] if RUN_BOTH_ADAPTIVE_RHO else [0]

results = []

for adaptive_rho in adaptive_rho_list:
    results.append(run_hw(adaptive_rho))

# =====================================================================
# 9. Summary
# =====================================================================

if RUN_BOTH_ADAPTIVE_RHO and len(results) == 2:

    off = results[0]
    on = results[1]

    print("\n=== Summary ===")

    print(
        f"ADMM iterations: "
        f"off={off['admm_iters']} | on={on['admm_iters']}"
    )

    print(
        f"Primal residual: "
        f"off={off['r_prim']:.3e} | on={on['r_prim']:.3e}"
    )

    print(
        f"Dual residual: "
        f"off={off['r_dual']:.3e} | on={on['r_dual']:.3e}"
    )

    print(
        f"Violation: "
        f"off={off['viol']:.3e} | on={on['viol']:.3e}"
    )

    print(
        f"Objective: "
        f"off={off['obj']:.6e} | on={on['obj']:.6e}"
    )

    print(
        f"HW time (ms): "
        f"off={off['hw_ms']:.3f} | on={on['hw_ms']:.3f}"
    )

# =====================================================================
# 10. Cleanup
# =====================================================================

buffers = [
    A_row_words,
    A_val_words,
    AT_row_words,
    AT_val_words,
    P_row_words,
    P_val_words,
    A_col_ptr_hw,
    AT_col_ptr_hw,
    P_col_ptr_hw,
    P_diag_hw,
    l_in_hw,
    u_in_hw,
    q_in_hw,
    rho_in_hw,
    x_out_hw,
    y_out_hw
]

for buf in buffers:
    buf.freebuffer()

print("\nBuffers released.")